In [14]:
import pandas as pd
df = pd.read_csv('data/updated_densitymap_hourly_cleaned.csv')

In [15]:
df.head()

,date,hour,building_floor,building_desc,latitude,longitude,building_type,density_cnt,capacity,dc_ratio
0,2021-03-31,1,4,209 BAY STATE ROAD,42.3507,-71.1028,Residential,1,8,0.13
1,2021-03-31,1,2,200-202 BAY STATE ROAD,42.3503,-71.1026,Residential,1,12,0.08
2,2021-03-31,1,2,184 BAY STATE ROAD,42.3503,-71.1019,Residential,1,6,0.17
3,2021-03-31,1,3,190 BAY STATE ROAD,42.3503,-71.1021,Residential,1,9,0.11
4,2021-03-31,1,3,161 BAY STATE ROAD,42.3506,-71.1009,Residential,1,5,0.20


In [16]:
bottle_fill = pd.read_csv('data/bottle_fillers.csv')
bottle_fill.head()

,Address A,Address B,Floor,Description,Accessibility,Handicap Accessible,Space@Bu room,Quantity,Date Installed,Type,Walk-Thru Notes,Completed
0,100,Ashford St,First,"Hallway across from vending machines, across f...",Public,Yes,191,1.0,NaN,Combo Bottle Filler/Drinking Fountain,two additional drinking fountain locations in ...,1/5/2024
1,120,Ashford St,First,"Kitchenette, Room 116A",Restricted,Yes,116A,1.0,NaN,Office-Style Water Cooler,NaN,1/5/2024
2,120,Ashford St,Second,"Kitchenette, Room 222",Restricted,Yes,222,1.0,NaN,Office-Style Water Cooler,NaN,1/5/2024
3,8,St. Mary's Street,Basement,Hallway by bathrooms,NaN,NaN,B81,1.0,Dec-23,Combo Bottle Filler/Drinking Fountain,NaN,NaN
4,8,St. Mary's Street,First,Hallway by electrical room,NaN,NaN,181,1.0,NaN,NaN,NaN,NaN


In [17]:
# Remove addresses that are nan

df = df.dropna(subset=['building_desc'])
bottle_fill = bottle_fill.dropna(subset=['Address A', 'Address B'])

In [18]:
average_counts = df.groupby('building_desc')[['capacity', 'dc_ratio']].mean()
print(len(average_counts))
average_counts.head()

243


,capacity,dc_ratio
building_desc,,
1 BUSWELL STREET,3.755505,0.661792
1 UNIVERSITY ROAD,162.754299,0.116412
10 BUICK STREET,54.114409,0.275676
10 COTTAGE FARM ROAD,0.000000,0.000000
10 LENOX STREET,32.831234,0.180870


In [19]:
import re

def average_address_if_range(address):
    ''' Takes in a range of addresses and returns the average address
        Ex: 20 - 30 Buswell St -> 25 Buswell St
    '''
    regex = r"(\d+)\s*-\s*(\d+)(.*)"
    parts = re.fullmatch(regex, address)
    if parts == None:
        return address

    start = int(parts.group(1))
    end = int(parts.group(2))
    street = parts.group(3)

    new = (start + end) // 2
    return str(new) + street

In [20]:
from functools import reduce

density_addresses = set(df.iloc[:, 3])
bottle_fill_address = set(bottle_fill['Address A'] + ' ' + bottle_fill['Address B'])

apply_transforms = lambda addresses, fns: [reduce(lambda acc, fn: fn(acc), fns, adr) for adr in addresses]

append_city = lambda address: address + ", Boston, MA"
transforms = [average_address_if_range, append_city]

density_addresses = apply_transforms(density_addresses, transforms)
bottle_fill_address = apply_transforms(bottle_fill_address, transforms)

density_addresses = list(set(density_addresses))
bottle_fill_address = list(set(bottle_fill_address))

print(density_addresses)
print(bottle_fill_address)


['700 COMMONWEALTH AVENUE, Boston, MA', '888 COMMONWEALTH AVENUE, Boston, MA', '214 BAY STATE ROAD, Boston, MA', '98 MOUNTFORT STREET, Boston, MA', '610 COMMONWEALTH AVENUE, Boston, MA', '838 COMMONWEALTH AVENUE, Boston, MA', '610 BEACON STREET, Boston, MA', '975 COMMONWEALTH AVENUE, Boston, MA', '50 WORTHINGTON ROAD, Boston, MA', '38 BUSWELL STREET, Boston, MA', '519 BEACON STREET, Boston, MA', "8 ST. MARY'S STREET, Boston, MA", '157 BAY STATE ROAD, Boston, MA', '175 BAY STATE ROAD, Boston, MA', '78 BAY STATE ROAD, Boston, MA', '67 BAY STATE ROAD, Boston, MA', '595 COMMONWEALTH AVENUE, Boston, MA', '171 AMORY STREET, Boston, MA', '180 BAY STATE ROAD, Boston, MA', '206 BAY STATE ROAD, Boston, MA', '3 BUSWELL STREET, Boston, MA', '178 BAY STATE ROAD, Boston, MA', '116 BAY STATE ROAD, Boston, MA', '48 BUSWELL STREET, Boston, MA', '75 CUMMINGTON MALL, Boston, MA', '41 BUSWELL STREET, Boston, MA', '153 BAY STATE ROAD, Boston, MA', '2 SILBER WAY, Boston, MA', '55 BUSWELL STREET, Boston, MA'

In [21]:
# Compute Longitude and latitude addresses
from geoapify import get_long_lat 

longitudes = []
latitudes = []

for address in bottle_fill_address:
    lat, lon = get_long_lat(address)
    longitudes += [lon]
    latitudes += [lat]


In [23]:
bottle_fill_coords_data = {
    'address': bottle_fill_address,
    'latitudes': latitudes,
    'longitudes': longitudes
}

bottle_fill_cords = pd.DataFrame(bottle_fill_coords_data)
bottle_fill_cords.head()
bottle_fill_cords.to_csv('data/bottle_filler_coordinates.csv')